# Code Attention manually and in PyTorch

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

## Simulate data and attention matrices

In [2]:
# parameters
n_batch = 4
n_embed = 10
context_length = 8
vocab_size = 40

# input data
data = torch.randint(vocab_size, (n_batch, context_length))
data

tensor([[36, 13, 23, 30, 16, 35,  2,  9],
        [17, 37, 19,  2,  8, 10, 27, 16],
        [33, 16,  0, 15, 13, 26, 17, 20],
        [37,  3, 24, 22, 12, 30, 11, 22]])

In [3]:
# embeddings matrix
embeddings = nn.Embedding(vocab_size, n_embed)

# create the q,k,v matrices
key = nn.Linear(n_embed, n_embed, bias=False)
query = nn.Linear(n_embed, n_embed, bias=False)
value = nn.Linear(n_embed, n_embed, bias=False)

## Process the data

In [4]:
# tokens to embeddings
x = embeddings(data)

# weight the data pre-attention
k = key(x)
q = query(x)
v = value(x)

In [5]:
# print data sizes
print(f'      Data matrix: {data.shape}')
print(f'Embeddings matrix: {embeddings.weight.shape}')
print(f' Token embeddings: {x.shape}')

# sizes of matrices
print('')
print(f'        Size of Q: {key.weight.shape}')
print(f'        Size of K: {key.weight.shape}')
print(f'        Size of V: {key.weight.shape}')

# print attention matrices sizes
print('')
print(f'     Size of Q(x): {k.shape}')
print(f'     Size of K(x): {q.shape}')
print(f'     Size of V(x): {v.shape}')

      Data matrix: torch.Size([4, 8])
Embeddings matrix: torch.Size([40, 10])
 Token embeddings: torch.Size([4, 8, 10])

        Size of Q: torch.Size([10, 10])
        Size of K: torch.Size([10, 10])
        Size of V: torch.Size([10, 10])

     Size of Q(x): torch.Size([4, 8, 10])
     Size of K(x): torch.Size([4, 8, 10])
     Size of V(x): torch.Size([4, 8, 10])


## Implement self-attention

In [6]:
### manual implementation

# "cosine similarity" between query and keys (note: would actually be cosine similarity if scaled by |q||k| )
qk = q @ k.transpose(-2,-1) # transpose non-batch dimensions

# variance scale the QK
qk_scaled = qk / (n_embed ** .5)

# apply mask for future tokens
pastmask = torch.tril(torch.ones(n_batch, context_length, context_length))
qk_scaled[pastmask==0] = -torch.inf # equivalent to adding a matrix of zeros/-infs

# softmaxify
qk_softmax = F.softmax(qk_scaled, dim=-1)

# and final attention mechanism
actsManual = qk_softmax @ v

print(f'Shape of activations (manual): {actsManual.shape}') # [batch, context, n_embed]

Shape of activations (manual): torch.Size([4, 8, 10])


In [7]:
### PyTorch implementation
actsTorch = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f'Shape of activations (PyTorch): {actsTorch.shape}')

Shape of activations (PyTorch): torch.Size([4, 8, 10])


In [8]:
# compare
print(actsManual[0,:,:])
print('')
print(actsTorch[0,:,:])
print('')
print(actsManual[0,:,:]-actsTorch[0,:,:])

print(f'\n\nAre they _exactly_ equal? {torch.equal(actsTorch,actsManual)}')
print(f'Are they "equal"? {torch.allclose(actsTorch,actsManual)}')

tensor([[ 0.2635, -0.5168,  0.5707,  0.9177,  0.9622,  0.7569, -0.4160, -0.0125,
          0.1783,  0.3748],
        [ 0.4639, -0.3409, -0.3997,  0.2029,  0.6211,  0.7801,  0.1959, -0.4679,
          0.1303,  0.7017],
        [ 0.4421, -0.1561, -0.6732, -0.0073,  0.3247,  0.6349,  0.2420, -0.6752,
          0.0049,  0.5528],
        [ 0.2577,  0.3683, -0.1720, -0.1743,  0.3765,  0.3375, -0.0151, -0.0065,
         -0.2825,  0.5678],
        [ 0.2692,  0.0451, -0.1098,  0.1585,  0.1861,  0.0909, -0.3040, -0.1049,
         -0.3434,  0.1833],
        [ 0.2479, -0.0017, -0.0655,  0.0871,  0.3078,  0.1931, -0.2013, -0.1005,
         -0.2596,  0.3320],
        [ 0.2484,  0.0103, -0.0804, -0.0034,  0.1628,  0.1296, -0.0935, -0.0954,
         -0.2548,  0.2835],
        [ 0.2873,  0.0253, -0.1470, -0.0274,  0.1432,  0.0466, -0.1176, -0.0075,
         -0.3055,  0.2919]], grad_fn=<SelectBackward0>)

tensor([[ 0.2635, -0.5168,  0.5707,  0.9177,  0.9622,  0.7569, -0.4160, -0.0125,
          0.1783, 

## CPU computation time

In [9]:
numReps = 50_000

# the manual version
start_time = time.time()
for _ in range(numReps):
    qk = q @ k.transpose(-2,-1) / (n_embed ** .5)
    pastmask = torch.tril(torch.ones(n_batch, context_length, context_length))
    qk[pastmask==0] = -torch.inf
    qk = F.softmax(qk, dim=-1)
    activations = qk @ v
print(f'---    Manual: {time.time()-start_time:.3f} sec')


# the optimized version
start_time = time.time()
for _ in range(numReps):
    activations = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f'--- Optimized: {time.time()-start_time:.3f} sec')

---    Manual: 5.304 sec
--- Optimized: 5.723 sec


## GPU computation time

In [10]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

## Using bigger matrices

In [11]:
# parameters
n_batch = 64
n_embed = 1000
context_length = 2048
vocab_size = 50257

# create matrices
data = torch.randint(vocab_size,(n_batch,context_length),dtype=torch.long,device=device)
embedding = nn.Embedding(vocab_size,n_embed,device=device)
key   = nn.Linear(n_embed,n_embed,bias=False,device=device)
query = nn.Linear(n_embed,n_embed,bias=False,device=device)
value = nn.Linear(n_embed,n_embed,bias=False,device=device)

x = embedding(data)
k = key(x)
q = query(x)
v = value(x)

## Now for the test!

In [12]:
numReps = 200

torch.cuda.synchronize() # synchronize the GPU&CPU. good for time-testing, bad for overall performance
start_time = time.time()

for _ in range(numReps):
    qk = q@k.transpose(-2,-1) * (n_embed**-.5)
    pastmask = torch.tril(torch.ones(n_batch,context_length,context_length,device=device))
    qk[pastmask==0] = -torch.inf
    qk = F.softmax(qk,dim=-1)
    activationsM = qk @ v
print(f'--- Manual:  {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
    activationsP = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'--- Pytorch: {time.time()-start_time:.3f} sec')

--- Manual:  25.332 sec
--- Pytorch: 48.032 sec


In [13]:
# some additional optimizations
import torch._dynamo
SDPA_compiled = torch.compile(F.scaled_dot_product_attention)
torch.set_float32_matmul_precision('high')

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [14]:
# FYI, FlashAttention: https://github.com/Dao-AILab/flash-attention

In [15]:
numReps = 200

torch.cuda.synchronize() # synchronize the GPU&CPU. good for time-testing, bad for overall performance
start_time = time.time()
for _ in range(numReps):
    qk = q@k.transpose(-2,-1) * (n_embed**-.5)
    pastmask = torch.tril(torch.ones(n_batch,context_length,context_length,device=device))
    qk[pastmask==0] = -torch.inf
    qk = F.softmax(qk,dim=-1)
    activationsM = qk @ v
print(f'--- Manual:  {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
    activationsP = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'--- Pytorch: {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
    activationsO = SDPA_compiled(q,k,v,is_causal=True)
print(f'--- Compiled: {time.time()-start_time:.3f} sec')


--- Manual:  27.297 sec
--- Pytorch: 48.134 sec


W1215 09:04:34.796000 291 torch/_inductor/utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode


--- Compiled: 8.039 sec
